# Urdu OCR for the Pakistan Code Urdu laws (Colab GPU)

Runs **EasyOCR 1.7.2 (`ur`+`en`)** on the 40 Urdu law PDFs with exactly the settings of the
project's benchmark (`src/ingestion/ocr_engine.py`: page rendered at 2x, `paragraph=True`),
but on a GPU instead of the laptop CPU (~100 s/page).

**Before you start**
1. *Runtime → Change runtime type → **T4 GPU*** → Save.
2. In Google Drive, create the folder **`urdu_ocr`** (in *My Drive*) and upload
   **`urdu_ocr_input.zip`** into it.
3. *Runtime → Run all*. Allow Drive access when asked.

Every page is saved to `My Drive/urdu_ocr/ocr_text/` as soon as it is done. If Colab
disconnects, just *Run all* again: finished pages are skipped.

At the end `urdu_ocr_output.zip` downloads (a copy is also saved in the Drive folder).
Then, in the project folder: `.\.venv\Scripts\python.exe main.py --import-ocr <path to urdu_ocr_output.zip>`


## 1. Check the GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then Run all again"
print("GPU:", torch.cuda.get_device_name(0))

## 2. Install EasyOCR (same version as the project) and PyMuPDF

In [ ]:
!pip -q install easyocr==1.7.2 pymupdf

## 3. Connect Google Drive and unpack the input

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import zipfile
from pathlib import Path

DRIVE_DIR = Path("/content/drive/MyDrive/urdu_ocr")
INPUT_ZIP = DRIVE_DIR / "urdu_ocr_input.zip"
OUT_DIR = DRIVE_DIR / "ocr_text"
WORK_DIR = Path("/content/input")

assert INPUT_ZIP.exists(), f"Upload urdu_ocr_input.zip to {DRIVE_DIR} first"
if not (WORK_DIR / "manifest.csv").exists():
    zipfile.ZipFile(INPUT_ZIP).extractall(WORK_DIR)
print(len(list((WORK_DIR / "pdfs").glob("*.pdf"))), "PDFs ready")

## 4. OCR all remaining pages (resumable)

In [ ]:
import csv, io, time
import fitz
import numpy as np
import easyocr
from PIL import Image

reader = easyocr.Reader(["ur", "en"], gpu=True)

manifest = sorted(csv.DictReader(open(WORK_DIR / "manifest.csv", encoding="utf-8")), key=lambda r: int(r["pages"]))
done_locally = set(open(WORK_DIR / "done_pages.txt", encoding="utf-8").read().split())

def out_path(doc_id, page):
    return OUT_DIR / doc_id / f"page_{page:04d}.txt"

todo = [(r["document_id"], p) for r in manifest for p in range(1, int(r["pages"]) + 1)
        if f"{r['document_id']}/page_{p:04d}.txt" not in done_locally and not out_path(r["document_id"], p).exists()]
print(f"{sum(int(r['pages']) for r in manifest)} pages in total, {len(todo)} still to OCR")

start, done = time.time(), 0
for row in manifest:
    pages = [p for doc_id, p in todo if doc_id == row["document_id"]]
    if not pages:
        continue
    with fitz.open(WORK_DIR / "pdfs" / row["filename"]) as pdf:
        for page in pages:
            pixmap = pdf[page - 1].get_pixmap(matrix=fitz.Matrix(2, 2), alpha=False)  # same as the project
            image = np.asarray(Image.open(io.BytesIO(pixmap.tobytes("png"))).convert("RGB"))
            lines = reader.readtext(image, detail=0, paragraph=True)
            target = out_path(row["document_id"], page)
            target.parent.mkdir(parents=True, exist_ok=True)
            tmp = target.with_suffix(".tmp")
            tmp.write_text("\n".join(str(line) for line in lines).strip(), encoding="utf-8")
            tmp.replace(target)
            done += 1
            if done % 25 == 0 or done == len(todo):
                rate = (time.time() - start) / done
                print(f"{done}/{len(todo)} pages | {rate:.1f} s/page | ~{(len(todo) - done) * rate / 60:.0f} min left")
print("OCR finished")

## 5. Zip the results and download

In [ ]:
import shutil
from google.colab import files

print(len(list(OUT_DIR.glob("*/page_*.txt"))), "page files in Drive")
archive = shutil.make_archive("/content/urdu_ocr_output", "zip", root_dir=OUT_DIR.parent, base_dir=OUT_DIR.name)
shutil.copy(archive, DRIVE_DIR / "urdu_ocr_output.zip")
files.download(archive)